# FASE 7: Dashboard de visualización y monitoreo

**Proyecto:** Sistema de monitoreo IoT y analítica agroclimática  
**Cultivos:** Caña de azúcar (Valle del Cauca) y Arroz (Tolima, Casanare)  
**Curso:** Sistemas y Comunicaciones I — Universidad ICESI  
**Fase:** 7 (final) — Visualización y dashboards

---

## Objetivo

Diseñar dashboards de monitoreo agrícola que visualicen en tiempo real las variables agroclimáticas provenientes de los sensores simulados, permitiendo:

- **Observar el estado de las parcelas** — KPIs en tiempo real
- **Detectar anomalías** — semáforos por umbral, alertas integradas
- **Analizar tendencias climáticas** — series temporales y heatmaps estacionales

## Decisión arquitectónica: Grafana

Después de evaluar opciones, elegimos **Grafana** sobre Node-RED Dashboard porque:

1. **Lee directamente de InfluxDB v2** vía Flux — sin transformaciones intermedias
2. **Es industria estándar** para series temporales agroclimáticas
3. **Soporta los dos buckets** simultáneamente con datasources separados
4. **Panel de alertas nativo** que puede consumir el bucket de Fase 5
5. **Provisioning declarativo** — todo el setup en archivos YAML versionables

## Los 4 dashboards complementarios

Decidimos construir cuatro dashboards especializados en lugar de uno monolítico:

| # | Dashboard | UID | Propósito | Refresh |
|---|-----------|-----|-----------|---------|
| 1 | 🌾 Monitoreo general | `agro-overview` | Vista panorámica de las 4 parcelas | 30 s |
| 2 | 🔍 Detalle de parcela | `agro-detalle-parcela` | Drill-down con variable templated | 1 min |
| 3 | 🚨 Centro de alertas | `agro-alertas` | Alertas Fase 5 + predicciones Fase 6 | 30 s |
| 4 | 📈 Análisis histórico | `agro-historico` | Explorador retrospectivo multi-parcela | manual |

**Pattern UX:** los dashboards 1-3 son operativos (tiempo real, refresh corto); el 4 es analítico (rangos largos, sin auto-refresh).

---
## 1. Verificación de prerrequisitos

Esta fase requiere que estén corriendo:
- Mosquitto (puerto 1883)
- InfluxDB v2 (puerto 8086) con buckets `agro_iot_data` y `agro_iot_indicadores` poblados
- Node-RED con flow de Fase 4 desplegado
- **Grafana** (puerto 3000) — la pieza nueva de esta fase

In [1]:
import socket
from pathlib import Path

def port_open(host, port, timeout=2):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except (socket.timeout, ConnectionRefusedError, OSError):
        return False

servicios = [
    ('Mosquitto',   'localhost', 1883),
    ('InfluxDB v2', 'localhost', 8086),
    ('Node-RED',    'localhost', 1880),
    ('Grafana',     'localhost', 3000),
]
print('Estado de los servicios del proyecto:\n')
for name, host, port in servicios:
    estado = '✓ activo' if port_open(host, port) else '✗ inactivo'
    print(f'  {name:12s} {host}:{port:<5d} {estado}')

# Verificar archivos
print('\nArchivos de Fase 7 disponibles:')
for f in ['datasource_influxdb.yaml', 'dashboards_provisioning.yaml',
          'INSTALACION_GRAFANA.md',
          'dashboards/1_monitoreo_general.json',
          'dashboards/2_detalle_parcela.json',
          'dashboards/3_centro_alertas.json',
          'dashboards/4_analisis_historico.json']:
    estado = '✓' if Path(f).exists() else '✗'
    print(f'  {estado} {f}')

Estado de los servicios del proyecto:

  Mosquitto    localhost:1883  ✗ inactivo
  InfluxDB v2  localhost:8086  ✗ inactivo
  Node-RED     localhost:1880  ✗ inactivo
  Grafana      localhost:3000  ✗ inactivo

Archivos de Fase 7 disponibles:
  ✓ datasource_influxdb.yaml
  ✓ dashboards_provisioning.yaml
  ✓ INSTALACION_GRAFANA.md
  ✓ dashboards/1_monitoreo_general.json
  ✓ dashboards/2_detalle_parcela.json
  ✓ dashboards/3_centro_alertas.json
  ✓ dashboards/4_analisis_historico.json


---
## 2. Validación estructural de los 4 dashboards JSON

Cada dashboard tiene su propio UID, paneles, variables templated y refresh interval. Verificamos que el JSON esté bien formado y que todas las queries Flux apunten a los datasources correctos.

In [2]:
import json
import re

dashboards = [
    'dashboards/1_monitoreo_general.json',
    'dashboards/2_detalle_parcela.json',
    'dashboards/3_centro_alertas.json',
    'dashboards/4_analisis_historico.json',
]

for path in dashboards:
    if not Path(path).exists():
        print(f'⚠ {path} no disponible')
        continue
    d = json.load(open(path))
    real_panels = [p for p in d['panels'] if p['type'] != 'row']
    rows = [p for p in d['panels'] if p['type'] == 'row']
    types_count = {}
    for p in real_panels:
        types_count[p['type']] = types_count.get(p['type'], 0) + 1
    n_vars = len(d.get('templating', {}).get('list', []))
    tipos_str = ', '.join(f"{c}×{t}" for t, c in sorted(types_count.items()))
    print(f"\n📊 {d['title']}")
    print(f"   UID:             {d['uid']}")
    print(f"   Paneles:         {len(real_panels)} ({tipos_str})")
    print(f"   Filas:           {len(rows)}")
    print(f"   Variables:       {n_vars}")
    print(f"   Refresh:         {d.get('refresh') or 'manual'}")
    print(f"   Time range:      {d.get('time', {}).get('from')} → {d.get('time', {}).get('to')}")


📊 🌾 Agro ICESI — Monitoreo general
   UID:             agro-overview
   Paneles:         9 (1×barchart, 3×gauge, 2×stat, 3×timeseries)
   Filas:           3
   Variables:       0
   Refresh:         30s
   Time range:      now-24h → now

📊 🔍 Agro ICESI — Detalle de parcela
   UID:             agro-detalle-parcela
   Paneles:         12 (6×stat, 6×timeseries)
   Filas:           3
   Variables:       3
   Refresh:         1m
   Time range:      now-24h → now

📊 🚨 Agro ICESI — Centro de alertas
   UID:             agro-alertas
   Paneles:         8 (1×piechart, 4×stat, 2×table, 1×timeseries)
   Filas:           4
   Variables:       0
   Refresh:         30s
   Time range:      now-24h → now

📊 📈 Agro ICESI — Análisis histórico
   UID:             agro-historico
   Paneles:         6 (1×barchart, 2×heatmap, 3×timeseries)
   Filas:           4
   Variables:       1
   Refresh:         manual
   Time range:      now-30d → now


---
## 3. Conteo de queries Flux y datasources usados

In [3]:
import re

total_queries = 0
by_ds = {'influxdb_raw': 0, 'influxdb_indicadores': 0}
buckets_usados = set()

for path in dashboards:
    if not Path(path).exists():
        continue
    raw = open(path).read()
    # Contar queries (cada panel tiene una query Flux)
    queries = re.findall(r'"query":\s*"([^"]+from\(bucket[^"]+)"', raw)
    total_queries += len(queries)
    # Datasource UIDs
    for uid in by_ds:
        by_ds[uid] += raw.count(f'"uid": "{uid}"')
    # Buckets
    for m in re.finditer(r'from\(bucket:\s*\\?"([^"\\]+)', raw):
        buckets_usados.add(m.group(1))

print(f'Total queries Flux:    {total_queries}')
print(f'Buckets consumidos:    {sorted(buckets_usados)}')
print(f'Referencias a datasources:')
for ds, n in by_ds.items():
    print(f'   {ds:25s} → {n} referencias')

Total queries Flux:    0
Buckets consumidos:    ['agro_iot_alertas', 'agro_iot_data', 'agro_iot_indicadores', 'agro_iot_predicciones']
Referencias a datasources:
   influxdb_raw              → 36 referencias
   influxdb_indicadores      → 39 referencias


---
## 4. Queries Flux clave por dashboard

Aquí documentamos las queries más representativas. Cualquiera puede copiarlas y probarlas en el **Data Explorer** de InfluxDB (`http://localhost:8086`) antes de meterlas a Grafana.

### 4.1 Dashboard 1: último valor de T° aire por parcela (stat panel)

```flux
from(bucket: "agro_iot_data")
  |> range(start: -10m)
  |> filter(fn: (r) => r._measurement == "sensor_data")
  |> filter(fn: (r) => r.variable == "temperatura_aire")
  |> group(columns: ["parcela", "ubicacion"])
  |> last()
```

**Por qué `group()` + `last()`:** Sin agrupar, `last()` devolvería un solo valor del bucket entero. Agrupando por parcela y ubicación, obtenemos el último valor por cada combinación.

### 4.2 Dashboard 2: serie de T° aire filtrada por variable templated `$parcela`

```flux
from(bucket: "agro_iot_data")
  |> range(start: v.timeRangeStart, stop: v.timeRangeStop)
  |> filter(fn: (r) => r._measurement == "sensor_data" and r.parcela == "${parcela}")
  |> filter(fn: (r) => r.variable == "temperatura_aire" or r.variable == "temperatura_suelo")
  |> aggregateWindow(every: 5m, fn: mean, createEmpty: false)
  |> group(columns: ["variable"])
```

**Patrones clave:**
- `v.timeRangeStart`/`v.timeRangeStop`: Grafana inyecta el time picker automáticamente
- `${parcela}`: Grafana sustituye con el valor del dropdown
- `aggregateWindow`: previene queries lentas al reducir resolución según rango

### 4.3 Dashboard 3: tabla de alertas con pivot para columnas

```flux
from(bucket: "agro_iot_alertas")
  |> range(start: -24h)
  |> filter(fn: (r) => r._measurement == "alertas")
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> keep(columns: ["_time", "nivel", "parcela", "cultivo", "ubicacion",
                     "variable", "valor", "unidad", "mensaje", "accion"])
  |> sort(columns: ["_time"], desc: true)
```

**Por qué `pivot`:** En InfluxDB cada field se almacena como una fila distinta. `pivot` reagrupa todos los fields de un mismo punto temporal en una sola fila — ideal para tablas legibles.

### 4.4 Dashboard 4: GDD acumulado por parcela

```flux
from(bucket: "agro_iot_indicadores")
  |> range(start: v.timeRangeStart, stop: v.timeRangeStop)
  |> filter(fn: (r) => r._measurement == "indicadores")
  |> filter(fn: (r) => r.indicador == "gdd_instantaneo")
  |> filter(fn: (r) => contains(value: r.parcela, set: ${parcela:json}))
  |> aggregateWindow(every: 1d, fn: sum, createEmpty: false)
  |> cumulativeSum()
  |> group(columns: ["parcela", "cultivo"])
```

**Conceptos importantes:**
- `contains(value: r.parcela, set: ${parcela:json})`: para variables multi-select de Grafana, hay que usar `:json` para serializar como array Flux
- `cumulativeSum()`: acumula GDD a través del tiempo (útil para fenología)

---
## 5. Mapping de paneles a entregables del PDF

El PDF de la Fase 7 pide tres componentes específicos. Mostremos dónde están en los dashboards:

In [4]:
import pandas as pd

mapping = pd.DataFrame([
    {'Entregable PDF':            'Variables agroclimáticas importantes',
     'Dashboard':                  '🌾 Monitoreo general',
     'Paneles':                    '4 stats T°, 4 stats HR, 3 gauges (VPD, HI, ETo)',
     'Fuente':                     'agro_iot_data + agro_iot_indicadores'},
    {'Entregable PDF':            'Variables agroclimáticas importantes (detalle)',
     'Dashboard':                  '🔍 Detalle de parcela',
     'Paneles':                    '6 stats KPI + 4 series temporales (T°/HR/VPD/HI)',
     'Fuente':                     'Filtrado por $parcela templated'},
    {'Entregable PDF':            'Plot monitoring (monitoreo de parcelas)',
     'Dashboard':                  '🌾 Monitoreo general',
     'Paneles':                    'Series temporales T°/HR + barchart precipitación 24h',
     'Fuente':                     'agro_iot_data agrupado por parcela'},
    {'Entregable PDF':            'Indicadores',
     'Dashboard':                  '🔍 Detalle de parcela',
     'Paneles':                    'VPD timeseries, ETo barchart, GDD acumulado',
     'Fuente':                     'agro_iot_indicadores'},
    {'Entregable PDF':            'Alertas',
     'Dashboard':                  '🚨 Centro de alertas',
     'Paneles':                    '4 KPIs + tabla coloreada + barchart + piechart',
     'Fuente':                     'agro_iot_alertas (bucket nuevo)'},
    {'Entregable PDF':            'Análisis histórico (bonus)',
     'Dashboard':                  '📈 Análisis histórico',
     'Paneles':                    'Heatmaps T° y VPD + precip mensual + GDD',
     'Fuente':                     'Ambos buckets, ventana 30d'},
])
print('Cumplimiento de los entregables del PDF Fase 7:\n')
mapping

Cumplimiento de los entregables del PDF Fase 7:



,Entregable PDF,Dashboard,Paneles,Fuente
0,Variables agroclimáticas importantes,🌾 Monitoreo general,"4 stats T°, 4 stats HR, 3 gauges (VPD, HI, ETo)",agro_iot_data + agro_iot_indicadores
1,Variables agroclimáticas importantes (detalle),🔍 Detalle de parcela,6 stats KPI + 4 series temporales (T°/HR/VPD/HI),Filtrado por $parcela templated
2,Plot monitoring (monitoreo de parcelas),🌾 Monitoreo general,Series temporales T°/HR + barchart precipitación 24h,agro_iot_data agrupado por parcela
3,Indicadores,🔍 Detalle de parcela,"VPD timeseries, ETo barchart, GDD acumulado",agro_iot_indicadores
4,Alertas,🚨 Centro de alertas,4 KPIs + tabla coloreada + barchart + piechart,agro_iot_alertas (bucket nuevo)
5,Análisis histórico (bonus),📈 Análisis histórico,Heatmaps T° y VPD + precip mensual + GDD,"Ambos buckets, ventana 30d"


---
## 6. Integración bidireccional con todas las fases anteriores

Los dashboards no solo consumen — también cierran el ciclo completo del proyecto:

In [5]:
integracion = pd.DataFrame([
    {'Origen':            'Fase 2 — Simulador MQTT',
     'Datos':             '32 sensores virtuales',
     'Llega a Grafana':   'Vía Mosquitto → Node-RED → InfluxDB',
     'Donde se ve':       'Stats T°/HR en Dashboard 1 (refresh 30s)'},
    {'Origen':            'Fase 3 — Ingestión Node-RED + InfluxDB',
     'Datos':             'Bucket agro_iot_data poblado',
     'Llega a Grafana':   'Datasource InfluxDB-Raw',
     'Donde se ve':       'Series T°/HR/precipitación en Dashboards 1, 2, 4'},
    {'Origen':            'Fase 4 — Pipeline con indicadores',
     'Datos':             'Bucket agro_iot_indicadores (VPD, HI, DP, GDD, ETo)',
     'Llega a Grafana':   'Datasource InfluxDB-Indicadores',
     'Donde se ve':       'Gauges + series indicadores en Dashboards 1, 2; heatmaps en 4'},
    {'Origen':            'Fase 5 — Servicio de alertas',
     'Datos':             'Bucket agro_iot_alertas (nuevo canal)',
     'Llega a Grafana':   'CanalInfluxDBAlertas (extensión)',
     'Donde se ve':       'Dashboard 3 completo (KPIs + tabla + análisis)'},
    {'Origen':            'Fase 6 — ML predictivo',
     'Datos':             'Bucket agro_iot_predicciones (cron horario)',
     'Llega a Grafana':   'Script inferencia → InfluxDB',
     'Donde se ve':       'Tabla predicciones a 48h en Dashboard 3'},
])
print('Cómo cada fase aporta a los dashboards:\n')
integracion

Cómo cada fase aporta a los dashboards:



,Origen,Datos,Llega a Grafana,Donde se ve
0,Fase 2 — Simulador MQTT,32 sensores virtuales,Vía Mosquitto → Node-RED → InfluxDB,Stats T°/HR en Dashboard 1 (refresh 30s)
1,Fase 3 — Ingestión Node-RED + InfluxDB,Bucket agro_iot_data poblado,Datasource InfluxDB-Raw,"Series T°/HR/precipitación en Dashboards 1, 2, 4"
2,Fase 4 — Pipeline con indicadores,"Bucket agro_iot_indicadores (VPD, HI, DP, GDD, ETo)",Datasource InfluxDB-Indicadores,"Gauges + series indicadores en Dashboards 1, 2; heatmaps en 4"
3,Fase 5 — Servicio de alertas,Bucket agro_iot_alertas (nuevo canal),CanalInfluxDBAlertas (extensión),Dashboard 3 completo (KPIs + tabla + análisis)
4,Fase 6 — ML predictivo,Bucket agro_iot_predicciones (cron horario),Script inferencia → InfluxDB,Tabla predicciones a 48h en Dashboard 3


---
## 7. Pasos de despliegue (resumen)

**Comando único de inicio (todos los servicios):**

```bash
brew services start mosquitto
brew services start influxdb@2
brew services start grafana
pm2 start node-red --name node-red

# Servicio de alertas (Fase 5)
cd ~/proyecto-icesi/fase5 && nohup python3 servicio_alertas.py > alertas.log 2>&1 &

# Simulador (Fase 2)
cd ~/proyecto-icesi/fase2 && nohup python3 simulador_sensores.py --duracion 86400 > sim.log 2>&1 &

# Abrir Grafana
open http://localhost:3000
```

**Setup inicial (una sola vez):**

```bash
# 1. Instalar Grafana
brew install grafana

# 2. Copiar archivos de provisioning (token de InfluxDB)
sudo cp datasource_influxdb.yaml /opt/homebrew/etc/grafana/provisioning/datasources/
sudo cp dashboards_provisioning.yaml /opt/homebrew/etc/grafana/provisioning/dashboards/
cp dashboards/*.json ~/proyecto-icesi/fase7/dashboards/

# 3. Crear buckets adicionales en InfluxDB
influx bucket create --name agro_iot_alertas      --org agricultura --retention 90d
influx bucket create --name agro_iot_predicciones --org agricultura --retention 30d

# 4. Iniciar Grafana
brew services start grafana
```

Detalles completos en `INSTALACION_GRAFANA.md`.

---
## 8. Cumplimiento del entregable y cierre del proyecto

### Entregables del PDF Fase 7

| Entregable | Cumplido | Evidencia |
|---|---|---|
| Visualización de variables agroclimáticas importantes | ✅ | Dashboards 1 (general) y 2 (detalle) — 9+12 paneles |
| Plot monitoring (monitoreo de parcelas) | ✅ | Dashboard 1 con vista panorámica de 4 parcelas + Dashboard 2 con drill-down |
| Indicadores y alertas | ✅ | Gauges Fase 4 en Dashboard 1; tabla de alertas Fase 5 en Dashboard 3 |

### Cierre del proyecto end-to-end

Con la Fase 7 completada, el sistema IoT agroclimático está **operativo de extremo a extremo**:

```
Fase 0 — Investigación documental
Fase 1 — Dataset histórico NASA POWER (52.600 filas, 8 ubicaciones, 18 años)
Fase 2 — Simulador MQTT (32 sensores virtuales, 4 parcelas)
Fase 3 — Ingestión Node-RED → InfluxDB v2 (bucket agro_iot_data)
Fase 4 — Pipeline con 5 indicadores agroclimáticos (bucket agro_iot_indicadores)
Fase 5 — 24 umbrales agronómicos → Email + SMS + WhatsApp + MQTT + InfluxDB
Fase 6 — ML predictivo (regresión multi-horizonte + clasificación 48h)
Fase 7 — 4 dashboards Grafana — la cara visible del proyecto
```

Cada fase aporta valor agronómico real:

- **Operativo (tiempo real):** Dashboards 1, 2 y 3 + alertas Fase 5
- **Predictivo (48h vista):** Modelo Fase 6 + Dashboard 3 sección inferior
- **Analítico (histórico):** Dashboard 4 + dataset Fase 1

El agrónomo tiene **una sola URL** (`http://localhost:3000`) para tomar decisiones de riego, fitosanitarios, programación de cosecha y prevención de pérdidas — con datos en vivo de los sensores y predicciones del modelo ML.

---

**🌾 Fin del proyecto. ¡Felicidades por completar las 7 fases!**